# Mapeamento de areais com Sentinel-2 e Random Forest

**Projeto final – Future Tech Leaders Brasil, Space Tech Bootcamp 2026**

Este notebook faz o fluxo completo do capstone, do limite da ottobacia até o mapa pronto para o QGIS:

| Etapa | O que faz | Inspirado em |
|---|---|---|
| 1 | Configuração e autenticação no Earth Engine | – |
| 2 | Área de estudo (ottobacia) | Polesia-Landcover (`geemap`) |
| 3 | Coleção Sentinel-2 de 2025 com máscara de nuvens | – |
| 4 | Índices por cena e variáveis intra-anuais | MapBiomas Pampa (medianas por estação) |
| 5 | Amostras de treino a partir dos seus polígonos | Polesia (pontos dentro de polígonos) |
| 6 | Comparação de modelos (Random Forest e outros) com validação cruzada espacial | Ebook FTL, aula Machine Learning I |
| 7 | Avaliação no conjunto de teste | – |
| 8 | Classificação da ottobacia e área por classe | MapBiomas (`smileRandomForest`) |
| 9 | Exportação para o QGIS (GeoTIFF e polígonos) | – |

**Antes de rodar, você precisa ter:**

1. Uma conta no Google Earth Engine vinculada a um projeto do Google Cloud.
2. O limite da ottobacia em shapefile ou GeoPackage, em `data/raw/`.
3. As amostras rotuladas em `data/samples/` (a etapa 5 explica o formato). Para o primeiro teste, as etapas 1 a 4 rodam sem amostras.

> O código deste notebook foi escrito para este projeto. Os repositórios citados são GPL-3.0: as ideias foram aproveitadas, mas nenhum trecho foi copiado, então você pode publicar este código com a licença que quiser (sugestão: MIT).

## 1. Configuração

Rode a célula de instalação só uma vez, no ambiente virtual do projeto (`.venv`).

In [1]:
# Instalação: rode uma vez no terminal, na raiz do repositório.
#   uv sync
# Isso cria o .venv com todas as bibliotecas do pyproject.toml.
# Depois selecione esse .venv como kernel do notebook no VS Code.
# Sem uv:
# %pip install -r ../requirements.txt

In [2]:
import ee
import geemap
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ID do seu projeto no Google Cloud (aparece no console do Earth Engine)
PROJETO_GCP = "SEU-PROJETO-GCP"

# Na primeira vez, abre o navegador para você autorizar o acesso.
# Depois disso, a autorização fica salva e esta linha pode ser comentada.
ee.Authenticate()
ee.Initialize(project=PROJETO_GCP)
print("Earth Engine conectado.")

EEException: Project 'projects/SEU-PROJETO-GCP' not found or deleted.

In [ ]:
# ---------------- Parâmetros do projeto (ajuste aqui) ----------------
ANO = 2025

# Caminhos relativos à pasta notebooks/
# Ottobacia Miracatu-Taquari (cod_otto 682636, nível 6, 588 km²), entre Manoel Viana e São Francisco de Assis
CAMINHO_BACIA = Path("../data/raw/area_de_estudo.shp")
CAMINHO_AMOSTRAS = Path("../data/samples/amostras_2025.gpkg")
PASTA_PROCESSADOS = Path("../data/processed")
PASTA_FIGURAS = Path("../figures")
PASTA_PROCESSADOS.mkdir(parents=True, exist_ok=True)
PASTA_FIGURAS.mkdir(parents=True, exist_ok=True)

# Sistema de referência das saídas: SIRGAS 2000 / UTM 21S.
# A bacia inteira (55,46° a 55,01° O) está no fuso 21. Use o mesmo CRS nos critérios do AHP.
CRS_SAIDA = "EPSG:31981"
ESCALA = 10  # metros

# Máscara de nuvens: pixels com Cloud Score+ abaixo deste valor são descartados (0 a 1).
LIMIAR_NUVEM = 0.60

# Regra de "solo exposto" em cada cena: NDVI baixo e BSI positivo.
LIMIAR_NDVI_EXPOSTO = 0.25

# Classes do mapa. O código (número) é o valor do pixel no GeoTIFF final.
CLASSES = {1: "Areal", 2: "Campo", 3: "Lavoura", 4: "Silvicultura", 5: "Água"}
CORES = {1: "#f2d16b", 2: "#9ccf7a", 3: "#e974ed", 4: "#7a5c3a", 5: "#2f6fd6"}

SEMENTE = 42  # garante que os sorteios sejam sempre os mesmos (reprodutibilidade)

## 2. Área de estudo

O limite da ottobacia é lido com `geopandas` e enviado ao Earth Engine com `geemap.gdf_to_ee`. O Earth Engine trabalha em coordenadas geográficas (EPSG:4326), por isso a reprojeção.

In [ ]:
bacia_gdf = gpd.read_file(CAMINHO_BACIA).to_crs("EPSG:4326")
bacia_gdf = bacia_gdf.dissolve()  # junta todas as partes num único polígono

bacia = geemap.gdf_to_ee(bacia_gdf)
geom = bacia.geometry()

area_km2 = geom.area(maxError=1).divide(1e6).getInfo()
print(f"Área da ottobacia: {area_km2:,.1f} km²")  # esperado: ~588 km² (campo area_total da ANA)

## 3. Coleção Sentinel-2 de 2025

- `COPERNICUS/S2_SR_HARMONIZED`: reflectância de superfície (já com correção atmosférica).
- `GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED`: nota de 0 a 1 por pixel, quanto maior, mais limpo. É hoje a máscara de nuvens mais confiável para Sentinel-2 no Earth Engine.

Em vez de usar uma imagem só, usamos **todas as cenas do ano**. É isso que permite medir o comportamento de cada pixel ao longo do tempo.

In [ ]:
BANDAS = ["B2", "B3", "B4", "B8", "B11", "B12"]  # azul, verde, vermelho, NIR, SWIR1, SWIR2

s2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(geom)
    .filterDate(f"{ANO}-01-01", f"{ANO + 1}-01-01")
)
cloud_score = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")


def preparar_cena(img):
    # Remove nuvens e sombras e converte os valores para reflectância (0 a 1).
    limpo = img.updateMask(img.select("cs_cdf").gte(LIMIAR_NUVEM))
    return (
        limpo.select(BANDAS)
        .multiply(0.0001)
        .copyProperties(img, ["system:time_start"])
    )


# linkCollection junta a nota de nuvem a cada cena correspondente
colecao = s2.linkCollection(cloud_score, ["cs_cdf"]).map(preparar_cena)
print("Cenas Sentinel-2 em", ANO, ":", colecao.size().getInfo())

## 4. Índices e variáveis intra-anuais

**Por cena**, calculamos três índices:

- **NDVI** = (NIR − Vermelho) / (NIR + Vermelho): vegetação.
- **NDWI** = (Verde − NIR) / (Verde + NIR): água.
- **BSI** = ((SWIR1 + Vermelho) − (NIR + Azul)) / ((SWIR1 + Vermelho) + (NIR + Azul)): solo exposto.

**Por pixel, no ano todo**, resumimos essas cenas em variáveis. A ideia central do projeto:

- o **areal** fica exposto o ano inteiro: `FREQ_EXPOSTO` alta, NDVI baixo em todas as estações;
- a **lavoura** fica exposta só no preparo do solo: `FREQ_EXPOSTO` intermediária e NDVI alto no verão (soja) ou no inverno (trigo, aveia).

Usamos o percentil 10 do NDVI em vez do mínimo, porque o mínimo costuma capturar uma nuvem ou sombra que escapou da máscara. As medianas por estação seguem a lógica do MapBiomas Pampa, que classifica com medianas de período seco e úmido.

> **Atenção para o mestrado:** não incluímos declividade, solo ou geologia como variáveis. Assim o inventário de areais fica independente dos critérios do AHP e pode ser usado para validá-lo depois sem circularidade.

In [ ]:
def adicionar_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    ndwi = img.normalizedDifference(["B3", "B8"]).rename("NDWI")
    bsi = img.expression(
        "((S1 + R) - (N + B)) / ((S1 + R) + (N + B))",
        {
            "S1": img.select("B11"),
            "R": img.select("B4"),
            "N": img.select("B8"),
            "B": img.select("B2"),
        },
    ).rename("BSI")
    # 1 = pixel com solo exposto nesta data, 0 = não
    exposto = ndvi.lt(LIMIAR_NDVI_EXPOSTO).And(bsi.gt(0)).rename("EXPOSTO").toFloat()
    return img.addBands([ndvi, ndwi, bsi, exposto])


colecao_idx = colecao.map(adicionar_indices)

# Mediana das bandas no ano (a "foto típica" do pixel)
medianas = colecao_idx.select(BANDAS).median().rename([f"{b}_med" for b in BANDAS])

# Distribuição do NDVI ao longo do ano: p10 (pior momento), p50 e p90 (melhor momento)
ndvi_pct = colecao_idx.select("NDVI").reduce(ee.Reducer.percentile([10, 50, 90]))
ndvi_desvio = colecao_idx.select("NDVI").reduce(ee.Reducer.stdDev()).rename("NDVI_desvio")

# Medianas por estação (verão: jan–mar; inverno: jun–set)
verao = colecao_idx.filter(ee.Filter.calendarRange(1, 3, "month"))
inverno = colecao_idx.filter(ee.Filter.calendarRange(6, 9, "month"))
ndvi_verao = verao.select("NDVI").median().rename("NDVI_verao")
ndvi_inverno = inverno.select("NDVI").median().rename("NDVI_inverno")

bsi_med = colecao_idx.select("BSI").median().rename("BSI_med")
ndwi_med = colecao_idx.select("NDWI").median().rename("NDWI_med")

# Variável-chave: fração das observações em que o pixel apareceu exposto (0 a 1)
freq_exposto = colecao_idx.select("EXPOSTO").mean().rename("FREQ_EXPOSTO")

# Número de observações válidas por pixel (controle de qualidade, não entra no modelo)
n_obs = colecao_idx.select("NDVI").count().rename("N_OBS")

variaveis = ee.Image.cat([
    medianas, ndvi_pct, ndvi_desvio, ndvi_verao, ndvi_inverno,
    bsi_med, ndwi_med, freq_exposto,
]).toFloat().clip(geom)

VARIAVEIS = variaveis.bandNames().getInfo()
print(len(VARIAVEIS), "variáveis:", VARIAVEIS)

### Teste visual

Este é o **teste de capacidade** que combinamos: se o mapa abaixo aparecer, o ambiente está pronto. Na camada `FREQ_EXPOSTO`, os areais devem aparecer em amarelo-claro (exposto quase o ano todo). Compare com a imagem de satélite do fundo e tente achar alguns.

In [ ]:
mapa = geemap.Map()
mapa.add_basemap("SATELLITE")
mapa.centerObject(bacia, 12)

mapa.addLayer(
    variaveis,
    {"bands": ["B4_med", "B3_med", "B2_med"], "min": 0.02, "max": 0.18},
    "Sentinel-2 cor natural (mediana 2025)",
)
mapa.addLayer(
    variaveis.select("NDVI_p10"),
    {"min": 0, "max": 0.7, "palette": ["#a6611a", "#f5f5f5", "#1a9641"]},
    "NDVI p10",
    False,
)
mapa.addLayer(
    variaveis.select("FREQ_EXPOSTO"),
    {"min": 0, "max": 1, "palette": ["#1b1b1b", "#7f3b08", "#fdb863", "#fff7bc"]},
    "Frequência de solo exposto",
)
mapa.addLayer(n_obs.clip(geom), {"min": 0, "max": 60, "palette": ["red", "yellow", "green"]}, "Nº de observações", False)
mapa.addLayer(bacia.style(color="white", fillColor="00000000", width=2), {}, "Ottobacia")
mapa

## 5. Amostras de treino

**Como criar no QGIS:**

1. Crie uma camada GeoPackage de **polígonos** (`data/samples/amostras_2025.gpkg`).
2. Adicione dois campos inteiros: `classe` (1 a 5, conforme `CLASSES`) e `poligono` (um número único por polígono).
3. Desenhe polígonos pequenos e "puros" sobre cada classe, usando o Google Satellite e a mediana Sentinel-2 como apoio. Meta: 30 a 50 polígonos por classe, bem espalhados pela bacia.

**Por que polígonos e não pontos soltos?** Pixels vizinhos são quase idênticos. Se um ficar no treino e o vizinho no teste, a acurácia sai inflada. Separando treino e teste **por polígono**, a avaliação fica honesta (é a autocorrelação espacial de que falamos).

O código sorteia pontos dentro de cada polígono (como no Polesia-Landcover) e extrai o valor das variáveis em cada ponto.

In [ ]:
PONTOS_POR_POLIGONO_MAX = 40


def pontos_em_poligonos(poligonos_gdf, max_pontos, semente=SEMENTE):
    # Sorteia pontos dentro de cada polígono, proporcional à área (no máximo max_pontos).
    # Cada ponto herda 'classe' e 'poligono'.
    proj = poligonos_gdf.to_crs(CRS_SAIDA)
    pixels_por_poligono = (proj.area / (ESCALA * ESCALA)).astype(int)
    n_pontos = np.clip(pixels_por_poligono // 2, 3, max_pontos)
    partes = []
    for (idx, linha), n in zip(proj.iterrows(), n_pontos):
        amostra = gpd.GeoSeries([linha.geometry], crs=proj.crs).sample_points(size=int(n), rng=semente + int(idx))
        pts = amostra.explode(index_parts=False)
        partes.append(gpd.GeoDataFrame(
            {"classe": int(linha["classe"]), "poligono": int(linha["poligono"])},
            index=range(len(pts)), geometry=pts.values, crs=proj.crs,
        ))
    return pd.concat(partes, ignore_index=True).to_crs("EPSG:4326")


amostras_gdf = gpd.read_file(CAMINHO_AMOSTRAS)
if amostras_gdf.geom_type.isin(["Polygon", "MultiPolygon"]).all():
    pontos_gdf = pontos_em_poligonos(amostras_gdf, PONTOS_POR_POLIGONO_MAX)
else:
    # Se você preferiu pontos, cada ponto vira o seu próprio "polígono" na validação
    pontos_gdf = amostras_gdf.to_crs("EPSG:4326").copy()
    if "poligono" not in pontos_gdf.columns:
        print("Aviso: sem coluna 'poligono'; a validação ficará otimista por causa de pontos vizinhos.")
        pontos_gdf["poligono"] = range(len(pontos_gdf))

print(pontos_gdf.groupby("classe").agg(pontos=("classe", "size"), poligonos=("poligono", "nunique"))
      .rename(index=CLASSES))

In [ ]:
# Envia os pontos ao Earth Engine e lê o valor de todas as variáveis em cada um
pontos_ee = geemap.gdf_to_ee(pontos_gdf[["classe", "poligono", "geometry"]])
amostrado = variaveis.sampleRegions(
    collection=pontos_ee,
    properties=["classe", "poligono"],
    scale=ESCALA,
    tileScale=4,
)

df = geemap.ee_to_df(amostrado)
df = df.dropna(subset=VARIAVEIS)  # descarta pontos sem imagem válida (ex.: borda da bacia)
df[["classe", "poligono"]] = df[["classe", "poligono"]].astype(int)
df.to_csv(PASTA_PROCESSADOS / f"amostras_variaveis_{ANO}.csv", index=False)
print(df.shape[0], "amostras com variáveis.")
df.groupby("classe")[["FREQ_EXPOSTO", "NDVI_p10", "NDVI_verao", "NDVI_inverno"]].median().rename(index=CLASSES).round(2)

A tabela acima já é uma **análise exploratória** útil para a apresentação: se a mediana de `FREQ_EXPOSTO` do areal for bem maior que a da lavoura, a variável está fazendo o trabalho dela.

## 6. Comparação de modelos com validação cruzada espacial

Esta etapa segue a estrutura da aula **Machine Learning I** do ebook do bootcamp (`ml-dl/supervised.ipynb`): um *baseline* Dummy, vários modelos clássicos em `Pipeline`, `GridSearchCV` com F1 macro e escolha do melhor na validação. A diferença é a validação: lá ela respeita o tempo (`TimeSeriesSplit`); aqui ela respeita o **espaço**.

1. Separamos cerca de 1/3 dos **polígonos** para teste, mantendo a proporção das classes (`StratifiedGroupKFold`). Esse teste só é usado no fim.
2. No restante, cada modelo tem seus hiperparâmetros ajustados com validação cruzada de 5 dobras, também por polígono.
3. A métrica é o **F1 macro**, a média do F1 de cada classe. Ela impede que o modelo "ganhe" acertando só as classes grandes (campo) e errando a pequena (areal).

A comparação vira o argumento da seção 3.4 do capstone (justificativa da escolha do modelo).

In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

METRICA = "f1_macro"

X = df[VARIAVEIS].to_numpy()
y = df["classe"].to_numpy()
grupos = df["poligono"].to_numpy()

# Separação treino/teste por polígono (~2/3 treino, ~1/3 teste)
divisor = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEMENTE)
idx_treino, idx_teste = next(divisor.split(X, y, grupos))
X_treino, y_treino, g_treino = X[idx_treino], y[idx_treino], grupos[idx_treino]
X_teste, y_teste = X[idx_teste], y[idx_teste]
print(f"Treino: {len(idx_treino)} pontos | Teste: {len(idx_teste)} pontos")

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)

# Baseline: o que um "modelo" que chuta a classe mais comum consegue?
baseline = DummyClassifier(strategy="most_frequent")
s_base = cross_val_score(baseline, X_treino, y_treino, groups=g_treino, cv=cv, scoring=METRICA)
print(f"Baseline (Dummy), {METRICA}: {s_base.mean():.3f} ± {s_base.std():.3f}")

In [ ]:
def pipeline(modelo):
    # Modelos sensíveis à escala das variáveis recebem padronização antes
    return Pipeline([("scaler", StandardScaler()), ("modelo", modelo)])


modelos = {
    "Regressão Logística": (
        pipeline(LogisticRegression(max_iter=5000)),
        {"modelo__C": [0.01, 0.1, 1, 10], "modelo__class_weight": [None, "balanced"]},
    ),
    "KNN": (
        pipeline(KNeighborsClassifier()),
        {"modelo__n_neighbors": [5, 11, 21], "modelo__weights": ["uniform", "distance"]},
    ),
    "SVM": (
        pipeline(SVC(kernel="rbf", random_state=SEMENTE)),
        {"modelo__C": [1, 10], "modelo__gamma": ["scale", 0.01], "modelo__class_weight": [None, "balanced"]},
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=SEMENTE),
        {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [2, 3]},
    ),
    # O Random Forest fica sem Pipeline para que os nomes dos hiperparâmetros
    # sejam os mesmos usados no Earth Engine (etapa 8).
    "Random Forest": (
        RandomForestClassifier(random_state=SEMENTE, n_jobs=-1),
        {"n_estimators": [100, 300], "max_features": ["sqrt", 0.5], "min_samples_leaf": [1, 3]},
    ),
}

buscas = {}
for nome, (estimador, grade) in modelos.items():
    busca_m = GridSearchCV(estimador, grade, cv=cv, scoring=METRICA, n_jobs=-1)
    busca_m.fit(X_treino, y_treino, groups=g_treino)
    buscas[nome] = busca_m
    print(f"{nome:<20} {METRICA} validação = {busca_m.best_score_:.3f}   {busca_m.best_params_}")

comparacao = pd.DataFrame({
    nome: {
        "F1 macro (média CV)": b.best_score_,
        "desvio": b.cv_results_["std_test_score"][b.best_index_],
        "melhores hiperparâmetros": b.best_params_,
    }
    for nome, b in buscas.items()
}).T.sort_values("F1 macro (média CV)", ascending=False)
comparacao.loc["Baseline (Dummy)"] = [s_base.mean(), s_base.std(), "-"]
comparacao.to_csv(PASTA_PROCESSADOS / "comparacao_modelos.csv")
comparacao

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ordem = comparacao.sort_values("F1 macro (média CV)")
ax.barh(ordem.index, ordem["F1 macro (média CV)"].astype(float),
        xerr=ordem["desvio"].astype(float), color="#4c72b0", capsize=3)
ax.set_xlabel("F1 macro na validação cruzada espacial")
ax.set_xlim(0, 1)
ax.set_title("Comparação de modelos")
plt.tight_layout()
fig.savefig(PASTA_FIGURAS / "comparacao_modelos.png", dpi=200)
plt.show()

# O mapa final usa o Random Forest, que roda nativamente no Earth Engine.
# Se outro modelo vencer com folga (diferença maior que o desvio), registre isso na discussão.
busca = buscas["Random Forest"]
melhor_nome = comparacao.drop("Baseline (Dummy)").index[0]
print(f"Melhor modelo na validação: {melhor_nome}")
print(f"Random Forest: {busca.best_score_:.3f} | hiperparâmetros: {busca.best_params_}")

## 7. Avaliação no conjunto de teste

Estas são as métricas que entram nas seções 3.5 e 3.6 do capstone.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

modelo = busca.best_estimator_
y_previsto = modelo.predict(X_teste)

nomes = [CLASSES[c] for c in sorted(CLASSES)]
print(classification_report(y_teste, y_previsto, labels=sorted(CLASSES), target_names=nomes, digits=3))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_teste, y_previsto, labels=sorted(CLASSES), display_labels=nomes,
    cmap="Blues", colorbar=False, ax=ax,
)
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe de referência")
ax.set_title(f"Matriz de confusão – teste ({ANO})")
plt.tight_layout()
fig.savefig(PASTA_FIGURAS / "matriz_confusao.png", dpi=200)
plt.show()

In [ ]:
# Importância por permutação: quanto o F1 cai quando embaralhamos cada variável.
# É mais confiável que a importância "de impureza" padrão do Random Forest.
imp = permutation_importance(
    modelo, X_teste, y_teste, scoring="f1_macro", n_repeats=20, random_state=SEMENTE, n_jobs=-1,
)
importancia = pd.Series(imp.importances_mean, index=VARIAVEIS).sort_values()

fig, ax = plt.subplots(figsize=(6, 5))
importancia.plot.barh(ax=ax, color="#4c72b0")
ax.set_xlabel("Queda média no F1 macro")
ax.set_title("Importância das variáveis")
plt.tight_layout()
fig.savefig(PASTA_FIGURAS / "importancia_variaveis.png", dpi=200)
plt.show()

## 8. Classificação da ottobacia

O mapa é gerado pelo Random Forest do próprio Earth Engine (`smileRandomForest`, o mesmo usado pelo MapBiomas), com os hiperparâmetros escolhidos na etapa 6 e treinado com os mesmos polígonos de treino. Assim o processamento da bacia inteira roda nos servidores do Google.

No fim conferimos a acurácia desse modelo nos polígonos de teste. Ela deve ficar próxima da obtida no scikit-learn.

In [ ]:
melhores = busca.best_params_
n_var = len(VARIAVEIS)
var_por_divisao = int(round(np.sqrt(n_var))) if melhores["max_features"] == "sqrt" else int(round(melhores["max_features"] * n_var))

poligonos_treino = sorted({int(g) for g in g_treino})
treino_ee = amostrado.filter(ee.Filter.inList("poligono", poligonos_treino))
teste_ee = amostrado.filter(ee.Filter.inList("poligono", poligonos_treino).Not())

classificador = ee.Classifier.smileRandomForest(
    numberOfTrees=melhores["n_estimators"],
    variablesPerSplit=var_por_divisao,
    minLeafPopulation=melhores["min_samples_leaf"],
    seed=SEMENTE,
).train(features=treino_ee, classProperty="classe", inputProperties=VARIAVEIS)

mapa_classes = variaveis.classify(classificador).rename("classe").toByte().clip(geom)

# Conferência no teste
matriz_ee = teste_ee.classify(classificador).errorMatrix("classe", "classification")
print(f"Acurácia global (GEE, teste): {matriz_ee.accuracy().getInfo():.3f}")
print(f"Kappa (GEE, teste): {matriz_ee.kappa().getInfo():.3f}")

In [ ]:
# Área de cada classe na ottobacia (km²)
area_por_classe = (
    ee.Image.pixelArea().divide(1e6)
    .addBands(mapa_classes)
    .reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName="classe"),
        geometry=geom, scale=ESCALA, maxPixels=1e10,
    )
    .get("groups")
    .getInfo()
)
tabela_area = pd.DataFrame(area_por_classe).rename(columns={"sum": "area_km2"})
tabela_area["classe_nome"] = tabela_area["classe"].map(CLASSES)
tabela_area["percentual"] = 100 * tabela_area["area_km2"] / tabela_area["area_km2"].sum()
tabela_area.to_csv(PASTA_PROCESSADOS / f"area_por_classe_{ANO}.csv", index=False)
tabela_area.round(2)

> A área "contada no mapa" ainda carrega os erros do classificador. Para a dissertação, vale corrigi-la com a matriz de confusão, pelo método de Olofsson et al. (2014). Para o capstone, a tabela acima já basta.

In [ ]:
codigos = sorted(CLASSES)
mapa2 = geemap.Map()
mapa2.add_basemap("SATELLITE")
mapa2.centerObject(bacia, 12)
mapa2.addLayer(
    mapa_classes,
    {"min": min(codigos), "max": max(codigos), "palette": [CORES[c].lstrip("#") for c in codigos]},
    f"Classificação {ANO}",
)
mapa2.addLayer(bacia.style(color="white", fillColor="00000000", width=2), {}, "Ottobacia")
mapa2.add_legend(title="Classes", legend_dict={CLASSES[c]: CORES[c] for c in codigos})
mapa2

## 9. Exportação para o QGIS

As exportações vão para uma pasta `arenizacao` no seu Google Drive (acompanhe na aba *Tasks* do Code Editor ou com `tarefa.status()`):

- `classes_2025.tif`: o mapa (1 = areal, 2 = campo, 3 = lavoura, 4 = silvicultura, 5 = água), em SIRGAS 2000 / UTM, 10 m.
- `variaveis_2025.tif`: todas as variáveis. `FREQ_EXPOSTO` e `NDVI_p10` podem virar camadas de apoio no mestrado.
- `areais_2025.shp`: os areais vetorizados, com a área de cada polígono em hectares. Use para validar o AHP depois.

In [ ]:
AREA_MIN_HA = 0.1  # descarta manchas menores que ~10 pixels (ruído)

tarefas = []

tarefas.append(ee.batch.Export.image.toDrive(
    image=mapa_classes,
    description=f"classes_{ANO}",
    folder="arenizacao",
    region=geom,
    scale=ESCALA,
    crs=CRS_SAIDA,
    maxPixels=1e10,
    fileFormat="GeoTIFF",
))

tarefas.append(ee.batch.Export.image.toDrive(
    image=variaveis,
    description=f"variaveis_{ANO}",
    folder="arenizacao",
    region=geom,
    scale=ESCALA,
    crs=CRS_SAIDA,
    maxPixels=1e10,
    fileFormat="GeoTIFF",
))

areais_vetor = (
    mapa_classes.eq(1).selfMask()
    .reduceToVectors(
        geometry=geom, scale=ESCALA, crs=CRS_SAIDA,
        geometryType="polygon", eightConnected=True, maxPixels=1e10,
    )
    .map(lambda f: f.set("area_ha", f.geometry().area(maxError=1).divide(1e4)))
    .filter(ee.Filter.gte("area_ha", AREA_MIN_HA))
)
tarefas.append(ee.batch.Export.table.toDrive(
    collection=areais_vetor,
    description=f"areais_{ANO}",
    folder="arenizacao",
    fileFormat="SHP",
))

for tarefa in tarefas:
    tarefa.start()
    print("Exportação iniciada:", tarefa.config["description"])

## Próximos passos

- **Comparação com a literatura:** De Nardin e Robaina (2009) mapearam cerca de 1.010 ha de areais na bacia do Arroio Miracatu (Landsat 2001 e IKONOS). Compare com a área da classe Areal acima, lembrando que método e resolução são diferentes.
- **Mapa web:** exportar `mapa2` para `docs/index.html` (notebook 04) e publicar pelo GitHub Pages.
- **Extensão temporal (opcional):** troque `ANO` para 2019 e rode de novo as etapas 3, 4, 8 e 9 com o mesmo classificador para comparar com 2025.
- **Mestrado:** abra `classes_2025.tif` e `areais_2025.shp` no QGIS, na mesma grade dos critérios do AHP.